# UCSD Mosaics EDA

Exploratory analysis of the HuggingFace-hosted UCSD Mosaics dataset
(`josauder/UCSD-mosaics-mirror`, GT-Clean variant; Edwards et al. 2017 +
Alonso et al. 2019 + Raine et al. 2024).

All inputs are read directly from HuggingFace via `datasets` and the
`UCSDMosaicsDataset` class — no S3, no manual downloads.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from mermaidseg.datasets.ucsd_mosaics import UCSDMosaicsDataset

## Instantiate the dataset

Default construction loads all 16 per-site HF splits (both `train` and `test`
rows of the source GT-Clean release) and fetches `classes.json` from the
mirror.

In [ ]:
dataset = UCSDMosaicsDataset()
print(f"Repo: {dataset.hf_repo_id}")
print(f"Sites loaded ({len(dataset.sites)}): {dataset.sites}")
print(f"Dataset length: {len(dataset)}")
print(f"Num source classes (incl. background): {dataset.num_source_classes}")

## Class table

`classes.json` from the mirror, sorted by ID. Pixel value `0` is the
*unlabeled / unidentified* ignore label and is not listed here (it is folded
into background by `UCSDMosaicsDataset`).

In [ ]:
class_df = pd.DataFrame(
    [
        {
            "id": int(e["id"]),
            "name": e["name"],
            "morphology": e["description"],
            "color_hex": "#{:02X}{:02X}{:02X}".format(*e["color_rgb"]),
            "color_rgb": tuple(e["color_rgb"]),
        }
        for e in dataset.class_table
    ]
).set_index("id")
class_df

## Per-site patch counts

Site IDs in this table are the HF split names (lowercased). The
`train` / `test` columns reflect the source GT-Clean `original_split`
field.

In [ ]:
sites_col = np.array(dataset.dataset["site"])
sites_lower = np.char.lower(sites_col.astype(str))
orig_split_col = np.array(dataset.dataset["original_split"])

per_site = (
    pd.DataFrame({"site": sites_lower, "original_split": orig_split_col})
    .pivot_table(index="site", columns="original_split", aggfunc="size", fill_value=0)
    .rename_axis(columns=None)
)
per_site["total"] = per_site.sum(axis=1)
per_site.loc["TOTAL"] = per_site.sum(axis=0)
per_site

## Sample image + colored dense mask per site

One randomly-chosen patch per site, with the dense uint8 mask rendered as a
translucent color overlay using the RGB palette from `classes.json`. The
legend on each panel shows up to 8 classes present in that patch.

In [ ]:
palette = np.zeros((max(int(e["id"]) for e in dataset.class_table) + 1, 3), dtype=np.uint8)
for entry in dataset.class_table:
    palette[int(entry["id"])] = entry["color_rgb"]
id2name = {0: "unlabeled / unidentified"} | {int(e["id"]): e["name"] for e in dataset.class_table}

sites_sorted = sorted(dataset.sites)
ncols = 4
nrows = int(np.ceil(len(sites_sorted) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

rng = np.random.default_rng(0)
for ax, site in zip(axes.ravel(), sites_sorted, strict=False):
    row_idxs = np.where(sites_lower == site)[0]
    if len(row_idxs) == 0:
        ax.axis("off")
        ax.set_title(f"{site}: no rows")
        continue
    idx = int(rng.choice(row_idxs))
    sample = dataset.dataset[idx]
    image = np.array(sample["image"])
    raw_mask = np.asarray(sample["label"], dtype=np.int64)
    color_mask = palette[raw_mask]

    ax.imshow(image)
    ax.imshow(color_mask, alpha=0.5)
    present_ids = sorted({int(v) for v in np.unique(raw_mask) if v != 0})[:8]
    handles = [
        Patch(facecolor=palette[cid] / 255.0, label=id2name[cid]) for cid in present_ids
    ]
    labelled_pct = float((raw_mask != 0).mean() * 100.0)
    ax.set_title(
        f"{site} / {sample['filename']}\n({labelled_pct:.1f}% labelled, "
        f"original_split={sample['original_split']})",
        fontsize=9,
    )
    ax.axis("off")
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=6, framealpha=0.7)

for ax in axes.ravel()[len(sites_sorted):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Class pixel frequency

Top 20 classes by share of labelled pixels, estimated from a random sample
of patches (sampling keeps the cell fast; the full dataset has ~1.2B label
pixels). The ignore-label fraction is reported separately at the bottom.

In [ ]:
N_SAMPLE = min(200, len(dataset))
sample_idxs = rng.choice(len(dataset), size=N_SAMPLE, replace=False)
pixel_counts = np.zeros(palette.shape[0], dtype=np.int64)
for idx in sample_idxs:
    raw_mask = np.asarray(dataset.dataset[int(idx)]["label"], dtype=np.int64)
    ids, counts = np.unique(raw_mask, return_counts=True)
    pixel_counts[ids] += counts

fg_counts = pixel_counts.copy()
fg_counts[0] = 0
fg_total = max(int(fg_counts.sum()), 1)

order = np.argsort(fg_counts)[::-1]
TOP = 20
top_ids = order[:TOP]
top_pct = fg_counts[top_ids] / fg_total * 100.0
top_names = [id2name[int(i)] for i in top_ids]
top_colors = [palette[int(i)] / 255.0 for i in top_ids]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(TOP), top_pct[::-1], color=top_colors[::-1])
ax.set_yticks(range(TOP))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel("% of labelled pixels")
ax.set_title(f"Top {TOP} classes by pixel share ({N_SAMPLE} sampled patches)")
ignore_pct = pixel_counts[0] / max(int(pixel_counts.sum()), 1) * 100.0
ax.text(
    0.98,
    0.02,
    f"ignore label = {ignore_pct:.1f}% of all pixels",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    color="gray",
)
plt.tight_layout()
plt.show()

## Sanity-check `__getitem__`

Confirm that `dataset[idx]` returns a `(image, source_labels)` pair where
`source_labels` is a dense integer mask in the **local** source-id space
(`0..N`, with `0` = background / ignore). By default the local IDs are
aligned 1:1 with the `classes.json` IDs, so the unique values seen here
should be a subset of `{0, 1, ..., 34}`.

In [ ]:
idx = 0
image, source_mask = dataset[idx]
sample = dataset.dataset[idx]
print(
    f"idx={idx} | filename={sample['filename']} | site={sample['site']} | "
    f"original_split={sample['original_split']}"
)
print(f"image shape: {image.shape}, dtype: {image.dtype}")
print(f"source_mask shape: {source_mask.shape}, dtype: {source_mask.dtype}")

unique_local = sorted({int(v) for v in np.unique(source_mask)})
expected_max = max(dataset.source_id2name) if dataset.source_id2name else 0
assert max(unique_local) <= expected_max, (
    f"mask contains local id {max(unique_local)} > expected max {expected_max}"
)
print(f"unique local mask ids: {unique_local}")

local_palette = np.zeros((dataset.num_source_classes, 3), dtype=np.uint8)
name_to_rgb = {e["name"]: e["color_rgb"] for e in dataset.class_table}
for local_id, name in dataset.source_id2name.items():
    if name in name_to_rgb:
        local_palette[local_id] = name_to_rgb[name]
local_color = local_palette[source_mask]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")
axes[1].imshow(image)
axes[1].imshow(local_color, alpha=0.6)
axes[1].set_title("Source-label mask overlay (local IDs)")
axes[1].axis("off")
plt.tight_layout()
plt.show()